# P2 — Kiểm định dữ liệu, split

Chủ sở hữu: **P2 Data Engineer**


In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')

    !pip -q install pyarrow mahotas lightgbm optuna shap imagehash
    !git clone -q https://github.com/AIVIETNAM-AIO-thanhnhan/car-parkinglot-count.git /content/code || (cd /content/code && git pull -q)

    SRC = '/content/code/src'
except ImportError:
    # Local: repo đã clone sẵn, deps đã cài qua requirements.txt
    from pathlib import Path
    SRC = str(Path.cwd().parent / 'src')

import sys; sys.path.insert(0, SRC)
import config, pklot_data, windows, features, evaluate_pklot
print('sẵn sàng —', 'Colab' if config.ON_COLAB else 'local')

sẵn sàng — local


## Ngày 1 — Quyết định `WINDOW_SIZE` & `SCALES` (P2 báo cáo, P1 duyệt)

Đo trên **CHỈ train+val** (`UFPR04`, `UFPR05`) — không đụng `PUCPR` (test), tránh leakage.
Camera cố định nên 1 ảnh đại diện/bãi là đủ (vị trí + kích thước ô giống hệt mọi ảnh cùng bãi).

In [2]:
import json

size_report = pklot_data.slot_size_report()
persp_report = pklot_data.perspective_report()

print('=== slot_size_report() ===')
print(json.dumps(size_report, indent=2))
print()
print('=== perspective_report() ===')
print(json.dumps(persp_report, indent=2))
print()
print(f"-> WINDOW_SIZE = {config.WINDOW_SIZE}  (p50 đo được: {size_report['w_p50']}, {size_report['h_p50']})")
print(f"-> SCALES = {config.SCALES}  (p90/p10 area đo được: {persp_report['worst_ratio']:.2f}x, "
      f"ngưỡng multi-scale = 2.0, needs_multiscale = {persp_report['needs_multiscale']})")

=== slot_size_report() ===
{
  "per_lot": {
    "UFPR04": {
      "n": 28,
      "w_p50": 86.0,
      "h_p50": 67.0
    },
    "UFPR05": {
      "n": 40,
      "w_p50": 97.5,
      "h_p50": 62.5
    }
  },
  "w_p50": 97.0,
  "h_p50": 64.5,
  "w_range": [
    59,
    178
  ],
  "h_range": [
    37,
    149
  ]
}

=== perspective_report() ===
{
  "per_lot": {
    "UFPR04": 3.454204204204204,
    "UFPR05": 3.3296793642093725
  },
  "worst_ratio": 3.454204204204204,
  "needs_multiscale": true
}

-> WINDOW_SIZE = 96  (p50 đo được: 97.0, 64.5)
-> SCALES = [0.5, 0.75, 1.0, 1.5, 2.0]  (p90/p10 area đo được: 3.45x, ngưỡng multi-scale = 2.0, needs_multiscale = True)
